In [14]:
# Step 1a: Create SparkSession and generate a larger dataset for timing experiments:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum, avg, desc, lit, when, round as spark_round
import time

spark = SparkSession.builder \
    .appName("M16-Lab03-Transformations-and-Actions") \
    .master("local[*]") \
    .getOrCreate()

print(f"✅ Spark {spark.version} running in local mode")


✅ Spark 4.0.2 running in local mode


In [15]:
# Step 1b: Generate a test dataset (100,000 rows)
import random

random.seed(42)

data = []
users = [f"U{i:04d}" for i in range(1, 501)]
songs = [f"S{i:03d}" for i in range(1, 51)]
artists = [f"A{i:02d}" for i in range(1, 11)]
statuses = ["completed", "completed", "completed", "skipped", "error"]

for i in range(100_000):
    data.append((
        f"P{i:06d}",
        random.choice(users),
        random.choice(songs),
        random.choice(artists),
        f"2025-03-{random.randint(1, 31):02d}",
        random.choice(statuses),
        random.randint(30, 300)
    ))

columns = ["play_id", "user_id", "song_id", "artist_id", "play_date", "status", "duration_seconds"]
df = spark.createDataFrame(data, columns)

print(f"✅ Test dataset created: {df.count()} rows, {len(df.columns)} columns")
df.printSchema()


✅ Test dataset created: 100000 rows, 7 columns
root
 |-- play_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- song_id: string (nullable = true)
 |-- artist_id: string (nullable = true)
 |-- play_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- duration_seconds: long (nullable = true)



In [16]:
## Step 2a: Chain transformations and time them:
print("=" * 60)
print("EXPERIMENT 1: Proving Transformations Are Lazy")
print("=" * 60)

start = time.time()

step1 = df.filter(col("status") == "completed")
step2 = step1.filter(col("duration_seconds") > 120)
step3 = step2.groupBy("artist_id").agg(
    count("play_id").alias("play_count"),
    spark_sum("duration_seconds").alias("total_seconds")
)
step4 = step3.withColumn("avg_seconds", spark_round(col("total_seconds") / col("play_count"), 1))
step5 = step4.orderBy(desc("play_count"))

elapsed = time.time() - start

print(f"\n5 transformations chained in {elapsed:.4f} seconds")
print(f"Type of step5: {type(step5)}")
print("Did Spark read the data? NO — these are just plans!")


EXPERIMENT 1: Proving Transformations Are Lazy

5 transformations chained in 0.1376 seconds
Type of step5: <class 'pyspark.sql.classic.dataframe.DataFrame'>
Did Spark read the data? NO — these are just plans!


In [17]:
# Step 2b: Now trigger an action and time it:
start = time.time()

step5.show(5)

elapsed = time.time() - start

print(f"\n.show() triggered execution in {elapsed:.4f} seconds")
print("NOW Spark read the data, filtered, grouped, aggregated, and sorted!")


+---------+----------+-------------+-----------+
|artist_id|play_count|total_seconds|avg_seconds|
+---------+----------+-------------+-----------+
|      A07|      4061|       854348|      210.4|
|      A02|      4032|       846346|      209.9|
|      A10|      4027|       853677|      212.0|
|      A05|      4007|       842347|      210.2|
|      A08|      3980|       836879|      210.3|
+---------+----------+-------------+-----------+
only showing top 5 rows

.show() triggered execution in 2.4090 seconds
NOW Spark read the data, filtered, grouped, aggregated, and sorted!


In [19]:
# Step 2c: Record your observations:
print("\n📋 OBSERVATIONS:")
print("1. Transformation chain time: 0.1376 seconds (near zero)")
print("2. Action (.show) time:       2.4090 seconds (much longer)")
print("3. This proves transformations are lazy — they build a plan, not results")



📋 OBSERVATIONS:
1. Transformation chain time: 0.1376 seconds (near zero)
2. Action (.show) time:       2.4090 seconds (much longer)
3. This proves transformations are lazy — they build a plan, not results


In [20]:
## Goal: Use .explain() to see what Spark plans to do WITHOUT executing.

#  Step 3a: Simple plan:
print("=" * 60)
print("EXPERIMENT 2: Inspecting Execution Plans")
print("=" * 60)

simple = df.filter(col("status") == "completed").select("play_id", "song_id", "duration_seconds")

print("\n--- Simple Plan (filter + select) ---")
simple.explain()


EXPERIMENT 2: Inspecting Execution Plans

--- Simple Plan (filter + select) ---
== Physical Plan ==
*(1) Project [play_id#187, song_id#189, duration_seconds#193L]
+- *(1) Filter (isnotnull(status#192) AND (status#192 = completed))
   +- *(1) Scan ExistingRDD[play_id#187,user_id#188,song_id#189,artist_id#190,play_date#191,status#192,duration_seconds#193L]




In [21]:
## Step 3b: Complex plan:
complex_query = (
    df.filter(col("status") == "completed")
      .filter(col("duration_seconds") > 60)
      .groupBy("artist_id")
      .agg(
          count("play_id").alias("play_count"),
          spark_sum("duration_seconds").alias("total_seconds")
      )
      .filter(col("play_count") > 100)
      .orderBy(desc("total_seconds"))
)

print("\n--- Complex Plan (filter + group + filter + sort) ---")
complex_query.explain()



--- Complex Plan (filter + group + filter + sort) ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_seconds#237L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_seconds#237L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=771]
      +- Filter (play_count#236L > 100)
         +- HashAggregate(keys=[artist_id#190], functions=[count(play_id#187), sum(duration_seconds#193L)])
            +- Exchange hashpartitioning(artist_id#190, 200), ENSURE_REQUIREMENTS, [plan_id=767]
               +- HashAggregate(keys=[artist_id#190], functions=[partial_count(play_id#187), partial_sum(duration_seconds#193L)])
                  +- Project [play_id#187, artist_id#190, duration_seconds#193L]
                     +- Filter ((isnotnull(status#192) AND isnotnull(duration_seconds#193L)) AND ((status#192 = completed) AND (duration_seconds#193L > 60)))
                        +- Scan ExistingRDD[play_id#187,user_id#188,song_id#189,artist_id#190,play_date#191,sta

In [22]:
## Step 3c: Extended plan (shows all stages):
print("\n--- Extended Plan (all optimization stages) ---")
complex_query.explain(True)



--- Extended Plan (all optimization stages) ---
== Parsed Logical Plan ==
'Sort ['total_seconds DESC NULLS LAST], true
+- Filter (play_count#236L > cast(100 as bigint))
   +- Aggregate [artist_id#190], [artist_id#190, count(play_id#187) AS play_count#236L, sum(duration_seconds#193L) AS total_seconds#237L]
      +- Filter (duration_seconds#193L > cast(60 as bigint))
         +- Filter (status#192 = completed)
            +- LogicalRDD [play_id#187, user_id#188, song_id#189, artist_id#190, play_date#191, status#192, duration_seconds#193L], false

== Analyzed Logical Plan ==
artist_id: string, play_count: bigint, total_seconds: bigint
Sort [total_seconds#237L DESC NULLS LAST], true
+- Filter (play_count#236L > cast(100 as bigint))
   +- Aggregate [artist_id#190], [artist_id#190, count(play_id#187) AS play_count#236L, sum(duration_seconds#193L) AS total_seconds#237L]
      +- Filter (duration_seconds#193L > cast(60 as bigint))
         +- Filter (status#192 = completed)
            +- Log

In [23]:
## Step 3d: Verify .explain() did NOT execute
start = time.time()
complex_query.explain()
explain_time = time.time() - start

start = time.time()
complex_query.show(5)
show_time = time.time() - start

print(f"\n.explain() time: {explain_time:.4f} seconds (plan only, no execution)")
print(f".show() time:    {show_time:.4f} seconds (full execution)")
print(f"\n✅ .explain() is safe — it shows the plan without running it")


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_seconds#237L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_seconds#237L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=771]
      +- Filter (play_count#236L > 100)
         +- HashAggregate(keys=[artist_id#190], functions=[count(play_id#187), sum(duration_seconds#193L)])
            +- Exchange hashpartitioning(artist_id#190, 200), ENSURE_REQUIREMENTS, [plan_id=767]
               +- HashAggregate(keys=[artist_id#190], functions=[partial_count(play_id#187), partial_sum(duration_seconds#193L)])
                  +- Project [play_id#187, artist_id#190, duration_seconds#193L]
                     +- Filter ((isnotnull(status#192) AND isnotnull(duration_seconds#193L)) AND ((status#192 = completed) AND (duration_seconds#193L > 60)))
                        +- Scan ExistingRDD[play_id#187,user_id#188,song_id#189,artist_id#190,play_date#191,status#192,duration_seconds#193L]


+---------+----------+

In [25]:
## Step 3e: In your notebook, annotate the .explain() output:
## What to look for in the plan:
# "FileScan" or "Scan" → reading data
# "Filter" → filtering rows
# "Project" → selecting columns (column pruning)
# "HashAggregate" → groupBy + aggregation
# "Sort" → orderBy
# "Exchange" → shuffle (data movement between nodes)


In [ ]:
## Step 3e: Annotating the .explain() Output

### Physical Plan Component Reference Table

| Plan Component | What It Means | Example from Complex Query | Performance Impact |
|:---------------|:--------------|:---------------------------|:-------------------|
| **Scan** | Reading data from source (RDD, file, or table). The starting point of any query. | `Scan ExistingRDD[play_id#0,user_id#1,song_id#2,artist_id#3,play_date#4,status#5,duration_seconds#6]` | I/O-bound — minimize data read with early filters |
| **Filter** | Row-level predicate pushdown. Removes rows that don't match conditions. | `Filter ((isnotnull(status#5) AND (status#5 = completed)) AND (duration_seconds#6 > 60))` | Reduces data early — apply filters as soon as possible |
| **Project** | Column pruning. Selects only the columns needed for subsequent operations. | Implicit in the Scan and later operations | Reduces memory footprint and shuffle size |
| **HashAggregate (partial)** | First stage of aggregation. Performs local aggregation on each partition before shuffle. | `partial_count(play_id#0), partial_sum(duration_seconds#6)` | Reduces data volume for shuffle — critical optimization |
| **Exchange** | **Shuffle operation**. Repartitions data across the cluster based on keys. Most expensive operation. | `Exchange hashpartitioning(artist_id#3, 200)` | Network I/O + disk spill — minimize if possible |
| **HashAggregate (final)** | Second stage of aggregation. Combines partial results from all partitions after shuffle. | `HashAggregate(keys=[artist_id#3], functions=[count(play_id#0), sum(duration_seconds#6)])` | CPU-bound — usually efficient |
| **Filter (post-aggregation)** | Filtering on aggregated results (HAVING clause equivalent). | `Filter (play_count#82L > 100)` | Cannot be pushed down — applies after grouping |
| **Sort** | Orders data by specified columns. Can trigger full data shuffle if not already partitioned correctly. | `Sort [total_seconds#84L DESC NULLS LAST], true, 0` | Memory/CPU intensive — use sparingly |




In [26]:
## Step 4a: Create a transformation chain:
print("=" * 60)
print("EXPERIMENT 3: Multiple Actions = Multiple Executions")
print("=" * 60)

pipeline = (
    df.filter(col("status") == "completed")
      .groupBy("song_id")
      .agg(count("play_id").alias("play_count"))
      .orderBy(desc("play_count"))
)


EXPERIMENT 3: Multiple Actions = Multiple Executions


In [27]:
## Step 4b: Trigger 3 actions and time each:
print("\nAction 1: .count()")
start = time.time()
total = pipeline.count()
t1 = time.time() - start
print(f"  Result: {total} songs | Time: {t1:.4f}s")

print("\nAction 2: .show()")
start = time.time()
pipeline.show(5)
t2 = time.time() - start
print(f"  Time: {t2:.4f}s")

print("\nAction 3: .collect()")
start = time.time()
result = pipeline.collect()
t3 = time.time() - start
print(f"  Result: {len(result)} rows | Time: {t3:.4f}s")

print(f"\n📋 TOTAL TIME: {t1 + t2 + t3:.4f}s")
print(f"Each action re-executed the FULL pipeline (read → filter → group → sort)")
print(f"The pipeline ran 3 TIMES!")



Action 1: .count()
  Result: 50 songs | Time: 1.5694s

Action 2: .show()
+-------+----------+
|song_id|play_count|
+-------+----------+
|   S034|      1279|
|   S022|      1260|
|   S003|      1249|
|   S006|      1245|
|   S048|      1237|
+-------+----------+
only showing top 5 rows
  Time: 1.4090s

Action 3: .collect()
  Result: 50 rows | Time: 1.3637s

📋 TOTAL TIME: 4.3421s
Each action re-executed the FULL pipeline (read → filter → group → sort)
The pipeline ran 3 TIMES!


In [28]:
## Step 5a: Same pipeline WITH caching:
print("=" * 60)
print("EXPERIMENT 4: Caching to Avoid Re-Execution")
print("=" * 60)

pipeline_cached = (
    df.filter(col("status") == "completed")
      .groupBy("song_id")
      .agg(count("play_id").alias("play_count"))
      .orderBy(desc("play_count"))
)

pipeline_cached.cache()
print("✅ .cache() called — results will be stored after first action")

print("\nAction 1: .count() (first action — full execution + cache)")
start = time.time()
total = pipeline_cached.count()
t1_cached = time.time() - start
print(f"  Result: {total} songs | Time: {t1_cached:.4f}s")

print("\nAction 2: .show() (reads from cache)")
start = time.time()
pipeline_cached.show(5)
t2_cached = time.time() - start
print(f"  Time: {t2_cached:.4f}s")

print("\nAction 3: .collect() (reads from cache)")
start = time.time()
result = pipeline_cached.collect()
t3_cached = time.time() - start
print(f"  Result: {len(result)} rows | Time: {t3_cached:.4f}s")

print(f"\n📋 TOTAL TIME WITH CACHE: {t1_cached + t2_cached + t3_cached:.4f}s")


EXPERIMENT 4: Caching to Avoid Re-Execution
✅ .cache() called — results will be stored after first action

Action 1: .count() (first action — full execution + cache)
  Result: 50 songs | Time: 3.0795s

Action 2: .show() (reads from cache)
+-------+----------+
|song_id|play_count|
+-------+----------+
|   S034|      1279|
|   S022|      1260|
|   S003|      1249|
|   S006|      1245|
|   S048|      1237|
+-------+----------+
only showing top 5 rows
  Time: 0.4487s

Action 3: .collect() (reads from cache)
  Result: 50 rows | Time: 0.6777s

📋 TOTAL TIME WITH CACHE: 4.2058s


In [29]:
## Step 5b: Compare with and without cache:
total_no_cache = t1 + t2 + t3
total_with_cache = t1_cached + t2_cached + t3_cached

print("\n" + "=" * 60)
print("CACHE COMPARISON")
print("=" * 60)
print(f"Without cache: {total_no_cache:.4f}s (3 full executions)")
print(f"With cache:    {total_with_cache:.4f}s (1 full + 2 cache reads)")
print(f"Speedup:       {total_no_cache / total_with_cache:.1f}x faster")

print("\n📋 KEY INSIGHT:")
print("Cache when a DataFrame is reused across multiple actions.")
print("Don't cache DataFrames used only once.")



CACHE COMPARISON
Without cache: 4.3421s (3 full executions)
With cache:    4.2058s (1 full + 2 cache reads)
Speedup:       1.0x faster

📋 KEY INSIGHT:
Cache when a DataFrame is reused across multiple actions.
Don't cache DataFrames used only once.


In [30]:
## Step 5c: Clean up the cache:
pipeline_cached.unpersist()
print("✅ Cache cleared")



✅ Cache cleared


In [31]:
## Complete this table in your notebook:
print("=" * 60)
print("EXPERIMENT 5: Classify 15 Operations")
print("=" * 60)

classifications = [
    ("df.filter(col('x') > 10)", "?"),
    ("df.select('a', 'b')", "?"),
    ("df.groupBy('x').count()", "?"),
    ("df.show()", "?"),
    ("df.count()", "?"),
    ("df.collect()", "?"),
    ("df.join(df2, 'key')", "?"),
    ("df.orderBy('x')", "?"),
    ("df.write.parquet('path')", "?"),
    ("df.withColumn('y', col('x') * 2)", "?"),
    ("df.distinct()", "?"),
    ("df.take(5)", "?"),
    ("df.explain()", "?"),
    ("df.union(df2)", "?"),
    ("df.first()", "?"),
]

print(f"\n{'Operation':<45} {'Your Answer':<15} {'Correct':<15}")
print("-" * 75)

answers = ["T", "T", "T", "A", "A", "A", "T", "T", "A", "T", "T", "A", "Neither", "T", "A"]

for (op, _), answer in zip(classifications, answers):
    print(f"{op:<45} {'___':<15} {answer:<15}")


EXPERIMENT 5: Classify 15 Operations

Operation                                     Your Answer     Correct        
---------------------------------------------------------------------------
df.filter(col('x') > 10)                      ___             T              
df.select('a', 'b')                           ___             T              
df.groupBy('x').count()                       ___             T              
df.show()                                     ___             A              
df.count()                                    ___             A              
df.collect()                                  ___             A              
df.join(df2, 'key')                           ___             T              
df.orderBy('x')                               ___             T              
df.write.parquet('path')                      ___             A              
df.withColumn('y', col('x') * 2)              ___             T              
df.distinct()               

In [32]:
## Verify your understanding:


## Step 6c: Classification Rules Reference Table

### Quick Classification Guide

| Rule | Description | Return Type | Execution | Examples |
|:-----|:------------|:------------|:----------|:---------|
| **Transformation** | Creates new DataFrame from existing one | DataFrame | **LAZY** (builds plan only) | `filter()`, `select()`, `groupBy()`, `join()`, `orderBy()`, `withColumn()`, `distinct()`, `union()`, `drop()`, `limit()` |
| **Action** | Triggers job execution, returns results to driver or writes data | Non-DataFrame (scalar, list, array, or Unit) | **EAGER** (executes immediately) | `show()`, `count()`, `collect()`, `take()`, `first()`, `head()`, `write()`, `save()`, `foreach()`, `reduce()` |
| **Special Operations** | Neither transformation nor action; metadata operations | Various | **NO EXECUTION** | `explain()`, `printSchema()`, `cache()`, `persist()`, `unpersist()`, `is_cached` |


### Memory & Performance Characteristics

| Operation Category | Memory Usage | CPU Usage | Network I/O | Disk I/O |
|:-------------------|:-------------|:----------|:------------|:---------|
| **Narrow Transformations** (filter, select, withColumn) | Low to Medium | Low | None | None |
| **Wide Transformations** (groupBy, join, repartition) | High | Medium | **SHUFFLE** | Spill to disk possible |
| **Actions** (collect, show, count) | Depends on result size | High | Driver transfer | Read/write |
| **Caching** (cache, persist) | Storage level dependent | Low | None | MEMORY/DISK |

### Decision Tree: Is It a Transformation or Action?

```text
                    START
                      │
                      ▼
        ┌─────────────────────────┐
        │ Does it return a        │
        │ DataFrame?              │
        └─────────────────────────┘
                      │
            ┌─────────┴─────────┐
          YES│                   │NO
            ▼                    ▼
    ┌───────────────┐   ┌─────────────────┐
    │ TRANSFORMATION│   │ Does it trigger │
    │ Lazy execution│   │ execution?      │
    │ Builds plan   │   └─────────────────┘
    └───────────────┘           │
                        ┌───────┴───────┐
                      YES│               │NO
                        ▼                ▼
                ┌─────────────┐   ┌─────────────┐
                │   ACTION    │   │   SPECIAL   │
                │ Eager exec  │   │ explain()   │
                │ Returns data│   │ printSchema │
                └─────────────┘   └─────────────┘







print("\n📋 CLASSIFICATION RULES:")
print("1. Returns a DataFrame → Transformation (lazy)")
print("2. Returns data to driver or writes to storage → Action (eager)")
print("3. .explain() is special — shows plan, does NOT execute")
print("4. .cache() is special — marks for caching, no execution")



📋 CLASSIFICATION RULES:
1. Returns a DataFrame → Transformation (lazy)
2. Returns data to driver or writes to storage → Action (eager)
3. .explain() is special — shows plan, does NOT execute
4. .cache() is special — marks for caching, no execution
